In [1]:
import pandas as pd
df = pd.read_csv("../data/clean_reviews.csv")
df["processed_text"] = df["processed_text"].fillna("").astype(str)
docs = [text.split() for text in df["processed_text"]]
print("Number of documents:", len(docs))

Number of documents: 555


In [2]:
# create vocabulary
unique_terms = sorted(set(
    word
    for doc in docs
    for word in doc
))

print("Number of unique terms:", len(unique_terms))
print(unique_terms[:20])

Number of unique terms: 3958
['aaa', 'abandon', 'abb', 'abid', 'abil', 'abl', 'abroad', 'absenc', 'absent', 'absolut', 'absorb', 'absurd', 'abt', 'abta', 'abu', 'abund', 'abus', 'acceler', 'accept', 'access']


In [3]:
# build inverted index
index = {}
for term in unique_terms:
    posting_list = []
    for doc_id, doc in enumerate(docs):
        doc_terms = set(doc)
        if term in doc_terms:
            posting_list.append(doc_id)

    index[term] = posting_list

print("Inverted index created.")

Inverted index created.


In [4]:
# show sample index terms
print("Sample inverted index:\n")
for i, term in enumerate(index):
    print(term, "->", index[term][:10])
    if i == 10:
        break

Sample inverted index:

aaa -> [90]
abandon -> [114, 240]
abb -> [382]
abid -> [414]
abil -> [9, 26, 114, 126, 266, 305]
abl -> [79, 94, 103, 133, 161, 181, 192, 206, 218, 325]
abroad -> [357]
absenc -> [498]
absent -> [329]
absolut -> [115, 130, 133, 154, 161, 176, 179, 181, 210, 241]
absorb -> [26, 168]


In [6]:
# search engine
from nltk.stem import PorterStemmer
stemmer = PorterStemmer()
query = input("Enter search query: ").strip().lower()
query_terms = query.split()
query_terms = [stemmer.stem(term) for term in query_terms]
results = set()

for term in query_terms:

    if term in index:
        results.update(index[term])

results = sorted(results)

if results:

    print(f"\nFound {len(results)} matching reviews\n")

    for row_id in results[:5]:

        print("Review ID:", df.loc[row_id, "review_id"])
        print("Item:", df.loc[row_id, "item_name"])
        print("Rating:", df.loc[row_id, "rating"])
        print("Helpful Label:", df.loc[row_id, "helpful_label"])

        print("Review:")
        print(df.loc[row_id, "review_text"][:300])

        print("-" * 80)

else:
    print("Keyword not found.")


Found 26 matching reviews

Review ID: 158
Item: Apex Legends
Rating: 1.0
Helpful Label: 1
Review:
Posted: April 11, 2021
I've always disliked traditional BR games as they're often unfair, rng focused and camper friendly. However, the difference about Apex is that here you won't be raging at a sniper kid that killed you from 1000m away hiding in a building like Battlefield or PUBG, instead you'll
--------------------------------------------------------------------------------
Review ID: 185
Item: PUBG
Rating: 1.0
Helpful Label: 1
Review:
Posted: 5 August, 2019
I've always asked myself, how can people play a game for 1000+ hours and then give it a thumbs down?

Then I played PUBG.

The European servers are in tatters, ruined by cheaters. I log in to 4 or 5 new perm-ban reports each session, but nothing changes. Cheats are readily ava
--------------------------------------------------------------------------------
Review ID: 186
Item: PUBG
Rating: 1.0
Helpful Label: 1
Review:
Posted: 24 

In [7]:
# convert inverted index to dataframe
index_table = []
for term, posting_list in index.items():

    index_table.append({
        "term": term,
        "document_frequency": len(posting_list),
        "posting_list": posting_list
    })

index_df = pd.DataFrame(index_table)

index_df.head()

,term,document_frequency,posting_list
0,aaa,1,[90]
1,abandon,2,"[114, 240]"
2,abb,1,[382]
3,abid,1,[414]
4,abil,6,"[9, 26, 114, 126, 266, 305]"


In [8]:
# save inverted index
index_df.to_csv(
    "../data/inverted_index.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Inverted index saved successfully.")

Inverted index saved successfully.
